In [0]:
# ============================================
# SILVER LAYER: Data Cleaning
# NO TIMESTAMP CONVERSION - Works 100%
# ============================================

print("🥈 SILVER LAYER: Data Cleaning")
print("=" * 50)

# Load bronze data
silver_df = spark.table("retail_lakehouse.bronze_online_retail")
starting_rows = silver_df.count()
print(f"📊 Starting rows: {starting_rows:,}")

🥈 SILVER LAYER: Data Cleaning
📊 Starting rows: 541,909


In [0]:
# Show raw data sample
print("👀 Raw data sample:")
silver_df.show(3, truncate=False)

👀 Raw data sample:
+---------+---------+----------------------------------+--------+----------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                       |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |
+---------+---------+----------------------------------+--------+----------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER|6       |01-12-2010 08:26|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN               |6       |01-12-2010 08:26|3.39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER    |8       |01-12-2010 08:26|2.75     |17850     |United Kingdom|
+---------+---------+----------------------------------+--------+----------------+---------+----------+--------------+
only showing top 3 rows


In [0]:
# Count nulls
from pyspark.sql.functions import col

print("🔍 Null values in raw data:")
print(f"  CustomerID: {silver_df.filter(col('CustomerID').isNull()).count()}")
print(f"  Description: {silver_df.filter(col('Description').isNull()).count()}")

🔍 Null values in raw data:
  CustomerID: 135080
  Description: 1454


In [0]:
# STEP 1: Remove null CustomerID
before = silver_df.count()
silver_df = silver_df.filter(col("CustomerID").isNotNull())
print(f"✅ Removed {before - silver_df.count():,} rows with no CustomerID")

✅ Removed 135,080 rows with no CustomerID


In [0]:
# STEP 2: Remove null Description
before = silver_df.count()
silver_df = silver_df.filter(col("Description").isNotNull())
print(f"✅ Removed {before - silver_df.count():,} rows with no Description")

✅ Removed 0 rows with no Description


In [0]:
# STEP 3: Remove duplicates
before = silver_df.count()
silver_df = silver_df.dropDuplicates()
print(f"✅ Removed {before - silver_df.count():,} duplicate rows")

✅ Removed 5,225 duplicate rows


In [0]:
# STEP 4: Keep only positive Quantity
before = silver_df.count()
silver_df = silver_df.filter(col("Quantity") > 0)
print(f"✅ Removed {before - silver_df.count():,} rows with Quantity <= 0")

✅ Removed 8,872 rows with Quantity <= 0


In [0]:
# STEP 5: Keep only positive UnitPrice
before = silver_df.count()
silver_df = silver_df.filter(col("UnitPrice") > 0)
print(f"✅ Removed {before - silver_df.count():,} rows with UnitPrice <= 0")
print(f"📊 Current rows: {silver_df.count():,}")

✅ Removed 40 rows with UnitPrice <= 0
📊 Current rows: 392,692


In [0]:
# STEP 6: Cast types (BUT NOT InvoiceDate!)
from pyspark.sql.functions import col

silver_df = silver_df.withColumn("CustomerID", col("CustomerID").cast("int"))
silver_df = silver_df.withColumn("Quantity", col("Quantity").cast("int"))
silver_df = silver_df.withColumn("UnitPrice", col("UnitPrice").cast("double"))

print("✅ Types cast (InvoiceDate left as string)")

✅ Types cast (InvoiceDate left as string)


In [0]:
# STEP 7: Add new columns (from string, not timestamp)
from pyspark.sql.functions import col, round

# Total amount
silver_df = silver_df.withColumn("TotalAmount", round(col("Quantity") * col("UnitPrice"), 2))

# Extract date parts from string using substr
# Format: "DD-MM-YYYY HH:MM"
silver_df = silver_df.withColumn("Year", col("InvoiceDate").substr(7, 4).cast("int"))
silver_df = silver_df.withColumn("Month", col("InvoiceDate").substr(4, 2).cast("int"))
silver_df = silver_df.withColumn("Day", col("InvoiceDate").substr(1, 2).cast("int"))

print("✅ New columns added: TotalAmount, Year, Month, Day")

✅ New columns added: TotalAmount, Year, Month, Day


In [0]:
# STEP 8: Preview cleaned data
print("👀 CLEANED DATA PREVIEW:")
silver_df.select("InvoiceNo", "Description", "Quantity", "UnitPrice", 
                 "TotalAmount", "Year", "Month", "Day", "Country").show(10)

👀 CLEANED DATA PREVIEW:
+---------+--------------------+--------+---------+-----------+----+-----+---+--------------+
|InvoiceNo|         Description|Quantity|UnitPrice|TotalAmount|Year|Month|Day|       Country|
+---------+--------------------+--------+---------+-----------+----+-----+---+--------------+
|   536365|WHITE HANGING HEA...|       6|     2.55|       15.3|2010|   12|  1|United Kingdom|
|   536365| WHITE METAL LANTERN|       6|     3.39|      20.34|2010|   12|  1|United Kingdom|
|   536365|CREAM CUPID HEART...|       8|     2.75|       22.0|2010|   12|  1|United Kingdom|
|   536365|KNITTED UNION FLA...|       6|     3.39|      20.34|2010|   12|  1|United Kingdom|
|   536365|RED WOOLLY HOTTIE...|       6|     3.39|      20.34|2010|   12|  1|United Kingdom|
|   536365|SET 7 BABUSHKA NE...|       2|     7.65|       15.3|2010|   12|  1|United Kingdom|
|   536365|GLASS STAR FROSTE...|       6|     4.25|       25.5|2010|   12|  1|United Kingdom|
|   536366|HAND WARMER UNION...|    

In [0]:
# STEP 9: Final quality check
print("✅ QUALITY CHECK:")
print(f"  Total rows: {silver_df.count():,}")
print(f"  Null CustomerID: {silver_df.filter(col('CustomerID').isNull()).count()}")
print(f"  Null Description: {silver_df.filter(col('Description').isNull()).count()}")
print(f"  Quantity <= 0: {silver_df.filter(col('Quantity') <= 0).count()}")
print(f"  UnitPrice <= 0: {silver_df.filter(col('UnitPrice') <= 0).count()}")
print(f"  Year range: {silver_df.agg({'Year': 'min'}).collect()[0][0]} - {silver_df.agg({'Year': 'max'}).collect()[0][0]}")

✅ QUALITY CHECK:
  Total rows: 392,692
  Null CustomerID: 0
  Null Description: 0
  Quantity <= 0: 0
  UnitPrice <= 0: 0
  Year range: 2010 - 2011


In [0]:
# STEP 10: Save Silver table
print("💾 SAVING SILVER TABLE...")

silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_lakehouse.silver_online_retail")

print("✅ Silver table saved!")

💾 SAVING SILVER TABLE...
✅ Silver table saved!


In [0]:
# STEP 11: Verify
verify_df = spark.table("retail_lakehouse.silver_online_retail")
print(f"🎯 VERIFICATION: {silver_df.count():,} rows saved")

print("\n📋 All tables:")
spark.sql("SHOW TABLES IN retail_lakehouse").show()

total_cleaned = silver_df.count()
print(f"\n🎉 SILVER LAYER COMPLETE!")
print(f"📊 Clean data: {total_cleaned:,} rows")
print(f"📊 Removed: {starting_rows - total_cleaned:,} bad rows")
print(f"\n📁 Next: 03_business_tables_gold")

🎯 VERIFICATION: 392,692 rows saved

📋 All tables:
+----------------+--------------------+-----------+
|        database|           tableName|isTemporary|
+----------------+--------------------+-----------+
|retail_lakehouse|bronze_online_retail|      false|
|retail_lakehouse|silver_online_retail|      false|
+----------------+--------------------+-----------+


🎉 SILVER LAYER COMPLETE!
📊 Clean data: 392,692 rows
📊 Removed: 149,217 bad rows

📁 Next: 03_business_tables_gold
